# Reasoning-model final-answer experiment
Evaluates Qwen3-0.6B on deterministic generated arithmetic, modular arithmetic and linear equations. Compares the model card's supported thinking and nonthinking template modes with identical questions and output-token ceilings. Only final integer exact match is scored. This does not establish internal reasoning, thought faithfulness, or performance on a standard benchmark. Pretrained runs are **pending user execution**.


## Runtime and provenance
Select a free GPU runtime manually. This notebook never provisions paid resources. Default models fit a typical T4; memory use varies. All pretrained weights download in your Colab runtime, not in checked-in validation. Each run saves resolved Hub commits, package versions, seeds, budgets, task IDs and outputs. The code resolves mutable `main` to one commit per model before loading. Save the archive before disconnecting; Colab storage is temporary.


In [ ]:
import sys, subprocess, os
from pathlib import Path
REPO = Path('/content/toollearn')
if not REPO.exists():
    subprocess.check_call(['git', 'clone', 'https://github.com/hiroshi-os/toollearn.git', str(REPO)])
os.chdir(REPO)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.57.6', 'peft==0.21.2', 'accelerate>=1,<2'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', '--no-deps'])
print('Source commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
import torch, platform, importlib.metadata
print({'python': platform.python_version(), 'torch': torch.__version__,
       'transformers': importlib.metadata.version('transformers'),
       'peft': importlib.metadata.version('peft'), 'cuda': torch.cuda.is_available()})
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU, then rerun. No provisioning is performed.'
free, total = torch.cuda.mem_get_info()
print(torch.cuda.get_device_name(0), 'free GiB:', round(free/2**30, 2), 'total GiB:', round(total/2**30, 2))
assert free >= 6*2**30, 'This modest default needs at least 6 GiB free. Restart runtime or reduce model/budgets.'
# This is a guard, not an exact memory estimate. OOM remains possible with larger settings.


## Configure held-out evaluation
Dev/test use disjoint numeric ranges and deterministic seeds. Use dev for prompt/budget tuning, then lock choices for test. A thinking response must close its thinking section and emit `FINAL: <integer>`; truncation is not credited for intermediate numbers. A 512-token thinking budget may truncate often; report budget-hit counts and invalid answers. Increasing it changes the experiment budget. Unsupported template mode flags raise an error instead of silently relabeling outputs.


In [ ]:
OUTPUT = 'artifacts/colab-reasoning'
MODEL, MODE, SPLIT = 'Qwen/Qwen3-0.6B', 'both', 'test'
SEED, COUNT, MAX_NEW_TOKENS = 17, 24, 512
subprocess.check_call([sys.executable, '-m', 'toollearn.reasoning', '--output', OUTPUT,
    '--model', MODEL, '--mode', MODE, '--split', SPLIT, '--count', str(COUNT),
    '--seed', str(SEED), '--max-new-tokens', str(MAX_NEW_TOKENS)])


In [ ]:
import json
for report_path in sorted(Path(OUTPUT).glob('*.json')):
    report = json.loads(report_path.read_text())
    if 'accuracy' in report:
        print(report_path.name, {k: report[k] for k in
            ['mode', 'tasks', 'accuracy', 'invalid_final_answers', 'budget_hit_count']})
# Full generated text, final-answer parsing, model identity, mode and token counts are saved.


In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive('/content/toollearn-results', 'zip', OUTPUT)
files.download(archive)
